# Taxi Fare Prediction Using Machine Learning

This notebook follows the assignment requirements for the **NYC Taxi Fare Prediction** dataset. It uses a feature-engineered **Artificial Neural Network** built with scikit-learn's `MLPRegressor`, then exposes the trained model through a Gradio interface.

## 1. Problem Overview

The goal is to predict taxi fare amounts from pickup and drop-off coordinates, passenger count, and time-based features derived from pickup datetime.

In [ ]:
import json
from pathlib import Path

import joblib
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from taxi_fare import load_dataset, prepare_training_frame, prepare_inference_frame

## 2. Load and Prepare the NYC Taxi Fare Prediction Dataset

Place the CSV file in `data/nyc_taxi_fare.csv` before running the training cells.

In [ ]:
data_path = Path('data/nyc_taxi_fare.csv')
raw = load_dataset(data_path, sample_size=200000, random_state=42)
features, target, stats = prepare_training_frame(raw)
stats

## 3. Train the Artificial Neural Network

The model uses `StandardScaler` followed by `MLPRegressor`, which satisfies the ANN requirement without using deep learning frameworks.

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42)
model = Pipeline([
    ('scaler', StandardScaler()),
    ('ann', MLPRegressor(
        hidden_layer_sizes=(128, 64, 32),
        activation='relu',
        solver='adam',
        batch_size=1024,
        learning_rate_init=0.001,
        max_iter=120,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=12,
        random_state=42
    ))
])
model.fit(x_train, y_train)
predictions = model.predict(x_test)
metrics = {
    'mae': mean_absolute_error(y_test, predictions),
    'rmse': mean_squared_error(y_test, predictions, squared=False),
    'r2': r2_score(y_test, predictions),
}
metrics

## 4. Save the Model

The exported file is used by the Gradio app and can also be uploaded to Hugging Face Spaces.

In [ ]:
Path('artifacts').mkdir(exist_ok=True)
joblib.dump(model, 'artifacts/taxi_fare_ann_model.joblib')
Path('artifacts/metrics.json').write_text(json.dumps(metrics, indent=2))
'artifacts/taxi_fare_ann_model.joblib'

## 5. Example Prediction

Use the same feature pipeline during inference to keep the app and training logic consistent.

In [ ]:
sample = prepare_inference_frame(
    pickup_datetime='2015-06-18 18:30:00',
    pickup_longitude=-73.985656,
    pickup_latitude=40.758896,
    dropoff_longitude=-73.971249,
    dropoff_latitude=40.7831,
    passenger_count=2,
)
float(model.predict(sample)[0])